In [1]:
import pandas as pd
import numpy as np
from IPython.display import display


In [3]:
df = pd.read_csv("data/hirisplex_full_41snps.csv")
print("Loaded SNP dictionary with", len(df), "rows")
display(df.head())


Loaded SNP dictionary with 90 rows


,rsid,gene,alleles,trait,trait_value
0,rs12913832,HERC2,AA,eye_color,blue
1,rs12913832,HERC2,AG,eye_color,intermediate
2,rs12913832,HERC2,GG,eye_color,brown
3,rs1800407,OCA2,CC,eye_color,brown_shift
4,rs1800407,OCA2,CT,eye_color,blue_shift


In [4]:
class HirisplexPredictor:

    def __init__(self, dataframe):
        self.table = dataframe.copy()

    def predict_trait_from_snp(self, rsid, genotype):
        genotype = genotype.upper()
        match = self.table[
            (self.table["rsid"] == rsid) & 
            (self.table["alleles"] == genotype)
        ]
        if match.empty:
            return None
        row = match.iloc[0]
        return row["trait"], row["trait_value"]

    def predict_all(self, snp_dict):
        results = {}

        for rsid, genotype in snp_dict.items():
            trait = self.predict_trait_from_snp(rsid, genotype)
            if trait:
                tname, tvalue = trait
                # group traits (e.g. hair may have multiple SNPs)
                results.setdefault(tname, []).append(tvalue)
        return results


In [5]:
EYE_MAP = {
    "blue": 0,
    "intermediate": 1,
    "brown": 2,
    
    "blue_shift": 0,
    "strong_blue_shift": 0,
    "light": 0,
    "medium": 1,
    "dark": 2,
}

HAIR_MAP = {
    "non_red": 0,
    "red_carrier": 1,
    "strong_red": 2,

    "dark": 0,
    "medium": 1,
    "light": 2,
    "brown": 0,
    "light_brown": 1,
    "dark_blonde": 1,
    "blonde": 2,
}

SKIN_MAP = {
    "light": 0,
    "medium": 1,
    "dark": 2,

    "darker": 2,
    "lighter": 0,
}

def most_common(lst):
    return max(set(lst), key=lst.count)

def encode_trait_vector(traits):
    eye = traits.get("eye_color", ["unknown"])
    hair = traits.get("hair_color", ["unknown"])
    skin = traits.get("skin_tone", ["unknown"])

    return {
        "eye_color": EYE_MAP.get(most_common(eye), -1),
        "hair_color": HAIR_MAP.get(most_common(hair), -1),
        "skin_tone": SKIN_MAP.get(most_common(skin), -1),
    }


In [6]:
predictor = HirisplexPredictor(df)
print("Predictor ready.")


Predictor ready.


In [7]:
sample_genotype = {
    "rs12913832": "AA",   # HERC2 → blue eyes
    "rs1805007": "CT",    # MC1R → red carrier
    "rs16891982": "GC",   # SLC45A2 → medium skin
}

sample_genotype


{'rs12913832': 'AA', 'rs1805007': 'CT', 'rs16891982': 'GC'}

In [8]:
pheno = predictor.predict_all(sample_genotype)
print("Predicted phenotype categories:")
pheno


Predicted phenotype categories:


{'eye_color': ['blue'], 'hair_color': ['red_carrier'], 'skin_tone': ['medium']}

In [9]:
trait_vector = encode_trait_vector(pheno)
print("Numeric trait vector:")
trait_vector


Numeric trait vector:


{'eye_color': 0, 'hair_color': 1, 'skin_tone': 1}

In [10]:
trait_vec_np = np.array([
    trait_vector["eye_color"],
    trait_vector["hair_color"],
    trait_vector["skin_tone"],
], dtype=np.float32)

trait_vec_np


array([0., 1., 1.], dtype=float32)

In [11]:
def genotype_to_trait_vector(genotype_dict):
    ph = predictor.predict_all(genotype_dict)
    tv = encode_trait_vector(ph)
    return np.array(list(tv.values()), dtype=np.float32)

genotype_to_trait_vector(sample_genotype)


array([0., 1., 1.], dtype=float32)